# GSE182616 data (old burn data)

Preprocessing burn data in 2 groups: mild/moderate and severe


In [1]:
from pathlib import Path
from io import StringIO
import re
import numpy as np
import pandas as pd

# -----------------------------
# Readers
# -----------------------------
def read_agilent_annotation(path):
    path = Path(path)
    df = pd.read_csv(path, sep="\t", comment="#", dtype=str, low_memory=False)
    df.columns = [c.strip() for c in df.columns]
    for c in df.columns:
        df[c] = df[c].astype(str).str.strip().str.strip('"')
        df.loc[df[c].isin(["nan", "NaN", "None"]), c] = ""
    return df

def read_geo_series_matrix_full(path):
    """
    Returns:
      - metadata_df with columns: key, values(list[str])
      - expr_df: ID_REF + GSM columns (strings)
    """
    path = Path(path)
    meta_rows = []
    table_lines = []
    in_table = False

    with path.open("r", encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.rstrip("\n")
            if line.startswith("!series_matrix_table_begin"):
                in_table = True
                continue
            if line.startswith("!series_matrix_table_end"):
                in_table = False
                continue

            if in_table:
                table_lines.append(line)
            elif line.startswith("!"):
                meta_rows.append(line)

    meta_parsed = []
    for row in meta_rows:
        parts = row.split("\t")
        key = parts[0].lstrip("!")
        values = parts[1:]
        meta_parsed.append((key, values))

    metadata_df = pd.DataFrame(
        {"key": [k for k, _ in meta_parsed], "values": [v for _, v in meta_parsed]}
    )

    if not table_lines:
        raise ValueError("No series matrix table found between begin/end markers.")

    expr_df = pd.read_csv(StringIO("\n".join(table_lines)), sep="\t", dtype=str, low_memory=False)
    expr_df.columns = [c.strip().strip('"') for c in expr_df.columns]
    for c in expr_df.columns:
        expr_df[c] = expr_df[c].astype(str).str.strip().str.strip('"')
        expr_df.loc[expr_df[c].isin(["nan", "NaN", "None"]), c] = ""

    return metadata_df, expr_df

# -----------------------------
# Metadata parsing helpers
# -----------------------------
def parse_kv_cell(s):
    if s is None:
        return None
    s = str(s).strip().strip('"')
    if ":" not in s:
        return None
    k, v = s.split(":", 1)
    return k.strip().lower(), v.strip()

def to_float(x):
    try:
        if x is None:
            return None
        xs = str(x).strip()
        if xs.lower() in {"na", "nan", ""}:
            return None
        xs = xs.replace("%", "")
        return float(xs)
    except Exception:
        return None

def assign_tbsa_bucket(tbsa_pct):
    if tbsa_pct is None:
        return None
    if tbsa_pct < 10:
        return "Mild"
    if 10 <= tbsa_pct < 20:
        return "Moderate"
    if 20 <= tbsa_pct <= 40:
        return "Severe"
    return "Massive"

def extract_sample_metadata(meta_df):
    geo_acc_rows = meta_df[meta_df["key"].str.lower() == "sample_geo_accession"]
    if geo_acc_rows.empty:
        raise ValueError("Could not find '!Sample_geo_accession' in series matrix metadata.")
    geo_samples = [str(x).strip().strip('"') for x in geo_acc_rows.iloc[0]["values"]]
    per_sample = {gsm: {} for gsm in geo_samples}

    char_rows = meta_df[
        meta_df["key"].str.lower().isin(["sample_characteristics_ch1", "sample_characteristics_ch2"])
    ]
    if char_rows.empty:
        out = pd.DataFrame(index=geo_samples)
        out.index.name = "sample_id"
        return out

    for _, row in char_rows.iterrows():
        values = row["values"]
        n = min(len(values), len(geo_samples))
        for j in range(n):
            gsm = geo_samples[j]
            kv = parse_kv_cell(values[j])
            if kv is None:
                continue
            k, v = kv
            per_sample[gsm][k] = v

    meta = pd.DataFrame.from_dict(per_sample, orient="index")
    meta.index.name = "sample_id"

    if "tbsa" in meta.columns:
        meta["tbsa_pct"] = meta["tbsa"].apply(to_float)

    return meta

def safe_label(x):
    x = str(x)
    x = re.sub(r"\s+", "_", x.strip())
    x = re.sub(r"[^A-Za-z0-9_\-]+", "", x)
    return x

# -----------------------------
# Expression preprocessing (same as yours)
# -----------------------------
def preprocess_burn_agilent(
    expr_df,
    anno_df,
    probe_col_expr="ID_REF",
    probe_col_anno="ID",
    gene_col_anno="GENE_SYMBOL",
    control_col_anno="CONTROL_TYPE",
    collapse="var",   # "var" | "mean" | "median"
    winsor_q=0.01,
    min_non_na_frac=0.95,
    min_var=1e-8,
    debug=True,
):
    expr = expr_df.copy()
    anno = anno_df.copy()

    if probe_col_expr not in expr.columns:
        raise ValueError(f"Expected '{probe_col_expr}' in expression table; got {expr.columns[:20].tolist()}")
    if probe_col_anno not in anno.columns:
        raise ValueError(f"Expected '{probe_col_anno}' in annotation table; got {anno.columns[:20].tolist()}")
    if gene_col_anno not in anno.columns:
        raise ValueError(f"Expected '{gene_col_anno}' in annotation table; got {anno.columns[:20].tolist()}")

    expr = expr.rename(columns={probe_col_expr: "probe_id"})
    anno = anno.rename(columns={probe_col_anno: "probe_id", gene_col_anno: "gene_symbol"})
    if control_col_anno in anno.columns:
        anno = anno.rename(columns={control_col_anno: "control_type"})

    keep_anno = ["probe_id", "gene_symbol"] + (["control_type"] if "control_type" in anno.columns else [])
    anno = anno[keep_anno].copy()

    expr["probe_id"] = expr["probe_id"].astype(str).str.strip().str.strip('"')
    anno["probe_id"] = anno["probe_id"].astype(str).str.strip().str.strip('"')
    anno["gene_symbol"] = anno["gene_symbol"].fillna("").astype(str).str.strip()

    merged = expr.merge(anno, on="probe_id", how="left")

    meta_cols = {"probe_id", "gene_symbol", "control_type"}
    sample_cols = [c for c in merged.columns if c not in meta_cols]

    for c in sample_cols:
        merged[c] = pd.to_numeric(merged[c], errors="coerce")

    merged["gene_symbol"] = merged["gene_symbol"].fillna("").astype(str).str.strip()
    merged = merged[merged["gene_symbol"].str.len() > 0]

    if "control_type" in merged.columns:
        ct = merged["control_type"].fillna("").astype(str).str.lower().str.strip()
        keep_ct = {"", "0", "false", "f"}
        merged = merged[ct.isin(keep_ct)]

    non_na_frac = merged[sample_cols].notna().mean(axis=1)
    merged = merged[non_na_frac >= min_non_na_frac]
    if merged.empty:
        raise RuntimeError("All rows were filtered out after missingness/controls filtering.")

    X = merged[["gene_symbol"] + sample_cols].copy()

    if collapse == "mean":
        gene_mat = X.groupby("gene_symbol")[sample_cols].mean()
    elif collapse == "median":
        gene_mat = X.groupby("gene_symbol")[sample_cols].median()
    elif collapse == "var":
        X["_var"] = X[sample_cols].var(axis=1, ddof=1)
        idx = X.groupby("gene_symbol")["_var"].idxmax()
        gene_mat = X.loc[idx].set_index("gene_symbol")[sample_cols]
    else:
        raise ValueError("collapse must be one of: var, mean, median")

    v = gene_mat.var(axis=1, ddof=1).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    gene_mat = gene_mat.loc[v > min_var]

    lo = gene_mat.quantile(winsor_q, axis=1)
    hi = gene_mat.quantile(1 - winsor_q, axis=1)
    gene_mat = gene_mat.apply(lambda row: row.clip(lo[row.name], hi[row.name]), axis=1)

    mu = gene_mat.mean(axis=1)
    sd = gene_mat.std(axis=1, ddof=1).replace(0, np.nan)
    gene_mat_z = gene_mat.sub(mu, axis=0).div(sd, axis=0).dropna(axis=0)

    if debug:
        print("[DEBUG] Final gene matrix:", gene_mat_z.shape, "(genes x samples)")

    return gene_mat_z

# -----------------------------
# TBSA-only grouping pipeline
# -----------------------------
def run_tbsa_only_pipeline(series_path, anno_path, out_dir, min_n=10, debug=True):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    print(f"[INFO] Loading annotation: {anno_path}")
    anno_df = read_agilent_annotation(anno_path)

    print(f"[INFO] Loading series matrix: {series_path}")
    meta_df, expr_df = read_geo_series_matrix_full(series_path)

    meta = extract_sample_metadata(meta_df)
    meta["tbsa_bucket"] = meta.get("tbsa_pct", pd.Series(index=meta.index, dtype=float)).apply(assign_tbsa_bucket)

    gene_mat_z = preprocess_burn_agilent(
        expr_df=expr_df,
        anno_df=anno_df,
        probe_col_expr="ID_REF",
        probe_col_anno="ID",
        min_non_na_frac=0.95,
        debug=debug,
    )

    # Align metadata to expression sample columns
    sample_cols = list(gene_mat_z.columns.astype(str))
    meta_aligned = meta.loc[meta.index.intersection(sample_cols)].copy()
    meta_aligned = meta_aligned.reindex(sample_cols)

    # Save aligned metadata
    out_meta = out_dir / "burn_sample_metadata_tbsa.tsv"
    meta_aligned.to_csv(out_meta, sep="\t")
    print(f"[SAVED] {out_meta}")

    keep_tbsa = ["Mild", "Moderate", "Severe", "Massive"]

    summary = []
    for tbsa_b in keep_tbsa:
        cols = meta_aligned.index[meta_aligned["tbsa_bucket"] == tbsa_b].tolist()

        # skip tiny groups if you want
        if len(cols) < min_n:
            print(f"[SKIP] {tbsa_b}: n={len(cols)} (<{min_n})")
            summary.append({"tbsa_bucket": tbsa_b, "n_samples": len(cols), "n_genes": int(gene_mat_z.shape[0]), "written": False})
            continue

        sub = gene_mat_z[cols]  # genes x samples
        out_path = out_dir / f"TBSA__{safe_label(tbsa_b)}__n{len(cols)}__zscored.tsv"
        sub.to_csv(out_path, sep="\t")
        print(f"[WRITE] {out_path.name}  shape={sub.shape} (genes x samples)")

        summary.append({"tbsa_bucket": tbsa_b, "n_samples": len(cols), "n_genes": int(sub.shape[0]), "written": True})

    summary_df = pd.DataFrame(summary).sort_values("tbsa_bucket")
    return summary_df

# -----------------------------
# Paths (notebook is in ./preprocessing/burn/)
# -----------------------------
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parents[1]  # ./ (BurnInjuries/)
BURN_DATA_DIR = PROJECT_ROOT / "burn_data"

series_path = BURN_DATA_DIR / "Burn_GSE182616_series_matrix.txt"
anno_path   = BURN_DATA_DIR / "BurnAgilentNotation_GPL17077-17467.txt"

out_dir = NOTEBOOK_DIR / "stratified"  # ./preprocessing/burn/preprocessed/

summary_df = run_tbsa_only_pipeline(
    series_path=series_path,
    anno_path=anno_path,
    out_dir=out_dir,
    min_n=1,      # set to 10 if you want to skip small TBSA groups
    debug=True,
)

display(summary_df)

[INFO] Loading annotation: /Users/zoeazra/Documents/CLS/Y2/Thesis/Implementation/BurnInjuries/burn_data/BurnAgilentNotation_GPL17077-17467.txt
[INFO] Loading series matrix: /Users/zoeazra/Documents/CLS/Y2/Thesis/Implementation/BurnInjuries/burn_data/Burn_GSE182616_series_matrix.txt
[DEBUG] Final gene matrix: (32080, 785) (genes x samples)
[SAVED] /Users/zoeazra/Documents/CLS/Y2/Thesis/Implementation/BurnInjuries/preprocessing/burn/preprocessed/burn_sample_metadata_tbsa_only.tsv
[WRITE] TBSA__Mild__n104__zscored.tsv  shape=(32080, 104) (genes x samples)
[WRITE] TBSA__Moderate__n259__zscored.tsv  shape=(32080, 259) (genes x samples)
[WRITE] TBSA__Severe__n292__zscored.tsv  shape=(32080, 292) (genes x samples)
[WRITE] TBSA__Massive__n130__zscored.tsv  shape=(32080, 130) (genes x samples)


,tbsa_bucket,n_samples,n_genes,written
3,Massive,130,32080,True
0,Mild,104,32080,True
1,Moderate,259,32080,True
2,Severe,292,32080,True


Preprocessing the groups based on Phases


In [13]:
from pathlib import Path
from io import StringIO
import re
import numpy as np
import pandas as pd

# -----------------------------
# Readers
# -----------------------------
def read_agilent_annotation(path):
    path = Path(path)
    df = pd.read_csv(path, sep="\t", comment="#", dtype=str, low_memory=False)
    df.columns = [c.strip() for c in df.columns]
    for c in df.columns:
        df[c] = df[c].astype(str).str.strip().str.strip('"')
        df.loc[df[c].isin(["nan", "NaN", "None"]), c] = ""
    return df

def read_geo_series_matrix_full(path):
    """
    Returns:
      - metadata_df with columns: key, values(list[str])
      - expr_df: ID_REF + GSM columns (strings)
    """
    path = Path(path)
    meta_rows = []
    table_lines = []
    in_table = False

    with path.open("r", encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.rstrip("\n")
            if line.startswith("!series_matrix_table_begin"):
                in_table = True
                continue
            if line.startswith("!series_matrix_table_end"):
                in_table = False
                continue

            if in_table:
                table_lines.append(line)
            elif line.startswith("!"):
                meta_rows.append(line)

    meta_parsed = []
    for row in meta_rows:
        parts = row.split("\t")
        key = parts[0].lstrip("!")
        values = parts[1:]
        meta_parsed.append((key, values))

    metadata_df = pd.DataFrame(
        {"key": [k for k, _ in meta_parsed], "values": [v for _, v in meta_parsed]}
    )

    if not table_lines:
        raise ValueError("No series matrix table found between begin/end markers.")

    expr_df = pd.read_csv(StringIO("\n".join(table_lines)), sep="\t", dtype=str, low_memory=False)
    expr_df.columns = [c.strip().strip('"') for c in expr_df.columns]
    for c in expr_df.columns:
        expr_df[c] = expr_df[c].astype(str).str.strip().str.strip('"')
        expr_df.loc[expr_df[c].isin(["nan", "NaN", "None"]), c] = ""

    return metadata_df, expr_df

# -----------------------------
# Metadata parsing helpers
# -----------------------------
def parse_kv_cell(s):
    if s is None:
        return None
    s = str(s).strip().strip('"')
    if ":" not in s:
        return None
    k, v = s.split(":", 1)
    return k.strip().lower(), v.strip()

def to_float(x):
    try:
        if x is None:
            return None
        xs = str(x).strip()
        if xs.lower() in {"na", "nan", ""}:
            return None
        xs = xs.replace("%", "")
        return float(xs)
    except Exception:
        return None

def parse_time_to_hours(raw):
    """
    Handles: hr0, hr4, hr24, hr108, D14, D21, etc.
    Returns hours as float.
    """
    if raw is None:
        return None
    t = str(raw).strip().strip('"').strip()
    t = t.replace("_", "").replace(" ", "").lower()

    m = re.match(r"^hr(\d+)$", t)
    if m:
        return float(m.group(1))

    m = re.match(r"^d(\d+)$", t)
    if m:
        return 24.0 * float(m.group(1))

    return None

def assign_time_bin(time_hours):
    if time_hours is None:
        return None
    if time_hours == 0:
        return "T0"
    if 1 <= time_hours < 24:
        return "Early"
    if 24 <= time_hours < 96:
        return "Mid"
    if 96 <= time_hours < 168:
        return "Late"
    if time_hours >= 169:  # D14 + D21
        return "FollowUp"
    return "Other"

def assign_phase_bucket(time_bin):
    if time_bin is None:
        return None
    if time_bin in {"T0", "Early", "Mid"}:
        return "Acute"
    if time_bin in {"Late"}:
        return "Proliferation"
    if time_bin in {"FollowUp"}:
        return "Remodelling"
    return None

def extract_sample_metadata(meta_df):
    geo_acc_rows = meta_df[meta_df["key"].str.lower() == "sample_geo_accession"]
    if geo_acc_rows.empty:
        raise ValueError("Could not find '!Sample_geo_accession' in series matrix metadata.")
    geo_samples = [str(x).strip().strip('"') for x in geo_acc_rows.iloc[0]["values"]]
    per_sample = {gsm: {} for gsm in geo_samples}

    char_rows = meta_df[
        meta_df["key"].str.lower().isin(["sample_characteristics_ch1", "sample_characteristics_ch2"])
    ]
    if char_rows.empty:
        out = pd.DataFrame(index=geo_samples)
        out.index.name = "sample_id"
        return out

    for _, row in char_rows.iterrows():
        values = row["values"]
        n = min(len(values), len(geo_samples))
        for j in range(n):
            gsm = geo_samples[j]
            kv = parse_kv_cell(values[j])
            if kv is None:
                continue
            k, v = kv
            per_sample[gsm][k] = v

    meta = pd.DataFrame.from_dict(per_sample, orient="index")
    meta.index.name = "sample_id"

    return meta

def safe_label(x):
    x = str(x)
    x = re.sub(r"\s+", "_", x.strip())
    x = re.sub(r"[^A-Za-z0-9_\-]+", "", x)
    return x

# -----------------------------
# Expression preprocessing (unchanged)
# -----------------------------
def preprocess_burn_agilent(
    expr_df,
    anno_df,
    probe_col_expr="ID_REF",
    probe_col_anno="ID",
    gene_col_anno="GENE_SYMBOL",
    control_col_anno="CONTROL_TYPE",
    collapse="var",
    winsor_q=0.01,
    min_non_na_frac=0.95,
    min_var=1e-8,
    debug=True,
):
    expr = expr_df.copy()
    anno = anno_df.copy()

    if probe_col_expr not in expr.columns:
        raise ValueError(f"Expected '{probe_col_expr}' in expression table; got {expr.columns[:20].tolist()}")
    if probe_col_anno not in anno.columns:
        raise ValueError(f"Expected '{probe_col_anno}' in annotation table; got {anno.columns[:20].tolist()}")
    if gene_col_anno not in anno.columns:
        raise ValueError(f"Expected '{gene_col_anno}' in annotation table; got {anno.columns[:20].tolist()}")

    expr = expr.rename(columns={probe_col_expr: "probe_id"})
    anno = anno.rename(columns={probe_col_anno: "probe_id", gene_col_anno: "gene_symbol"})
    if control_col_anno in anno.columns:
        anno = anno.rename(columns={control_col_anno: "control_type"})

    keep_anno = ["probe_id", "gene_symbol"] + (["control_type"] if "control_type" in anno.columns else [])
    anno = anno[keep_anno].copy()

    expr["probe_id"] = expr["probe_id"].astype(str).str.strip().str.strip('"')
    anno["probe_id"] = anno["probe_id"].astype(str).str.strip().str.strip('"')
    anno["gene_symbol"] = anno["gene_symbol"].fillna("").astype(str).str.strip()

    merged = expr.merge(anno, on="probe_id", how="left")

    meta_cols = {"probe_id", "gene_symbol", "control_type"}
    sample_cols = [c for c in merged.columns if c not in meta_cols]

    for c in sample_cols:
        merged[c] = pd.to_numeric(merged[c], errors="coerce")

    merged["gene_symbol"] = merged["gene_symbol"].fillna("").astype(str).str.strip()
    merged = merged[merged["gene_symbol"].str.len() > 0]

    if "control_type" in merged.columns:
        ct = merged["control_type"].fillna("").astype(str).str.lower().str.strip()
        keep_ct = {"", "0", "false", "f"}
        merged = merged[ct.isin(keep_ct)]

    non_na_frac = merged[sample_cols].notna().mean(axis=1)
    merged = merged[non_na_frac >= min_non_na_frac]
    if merged.empty:
        raise RuntimeError("All rows were filtered out after missingness/controls filtering.")

    X = merged[["gene_symbol"] + sample_cols].copy()

    if collapse == "mean":
        gene_mat = X.groupby("gene_symbol")[sample_cols].mean()
    elif collapse == "median":
        gene_mat = X.groupby("gene_symbol")[sample_cols].median()
    elif collapse == "var":
        X["_var"] = X[sample_cols].var(axis=1, ddof=1)
        idx = X.groupby("gene_symbol")["_var"].idxmax()
        gene_mat = X.loc[idx].set_index("gene_symbol")[sample_cols]
    else:
        raise ValueError("collapse must be one of: var, mean, median")

    v = gene_mat.var(axis=1, ddof=1).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    gene_mat = gene_mat.loc[v > min_var]

    lo = gene_mat.quantile(winsor_q, axis=1)
    hi = gene_mat.quantile(1 - winsor_q, axis=1)
    gene_mat = gene_mat.apply(lambda row: row.clip(lo[row.name], hi[row.name]), axis=1)

    mu = gene_mat.mean(axis=1)
    sd = gene_mat.std(axis=1, ddof=1).replace(0, np.nan)
    gene_mat_z = gene_mat.sub(mu, axis=0).div(sd, axis=0).dropna(axis=0)

    if debug:
        print("[DEBUG] Final gene matrix:", gene_mat_z.shape, "(genes x samples)")

    return gene_mat_z

# -----------------------------
# PHASE-only grouping pipeline
# -----------------------------
def run_phase_only_pipeline(series_path, anno_path, out_dir, min_n=10, debug=True):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    print(f"[INFO] Loading annotation: {anno_path}")
    anno_df = read_agilent_annotation(anno_path)

    print(f"[INFO] Loading series matrix: {series_path}")
    meta_df, expr_df = read_geo_series_matrix_full(series_path)

    # --- build metadata (including phase bins)
    meta = extract_sample_metadata(meta_df)

    # find the time-point field in metadata (it is usually "time point")
    time_col = None
    for cand in ["time point", "time_point", "timepoint", "time"]:
        if cand in meta.columns:
            time_col = cand
            break
    if time_col is None:
        raise KeyError(f"Could not find a time-point column in metadata. Available columns: {meta.columns.tolist()}")

    meta["time_point_raw"] = meta[time_col]
    meta["time_hours"] = meta["time_point_raw"].apply(parse_time_to_hours)
    meta["time_bin"] = meta["time_hours"].apply(assign_time_bin)
    meta["phase_bucket"] = meta["time_bin"].apply(assign_phase_bucket)

    if debug:
        print("\n[INFO] phase_bucket counts:")
        print(meta["phase_bucket"].value_counts(dropna=False))

    # --- preprocess expression into gene x sample z-scored matrix
    gene_mat_z = preprocess_burn_agilent(
        expr_df=expr_df,
        anno_df=anno_df,
        probe_col_expr="ID_REF",
        probe_col_anno="ID",
        min_non_na_frac=0.95,
        debug=debug,
    )

    # --- align metadata to expression sample columns
    sample_cols = list(gene_mat_z.columns.astype(str))
    meta_aligned = meta.reindex(sample_cols).copy()   # keeps same order as expression columns

    # Save aligned metadata
    out_meta = out_dir / "burn_sample_metadata_phase.tsv"
    meta_aligned.to_csv(out_meta, sep="\t")
    print(f"[SAVED] {out_meta}")

    keep_phases = ["Acute", "Proliferation", "Remodelling"]

    summary = []
    for ph in keep_phases:
        cols = meta_aligned.index[meta_aligned["phase_bucket"] == ph].tolist()

        if len(cols) < min_n:
            print(f"[SKIP] {ph}: n={len(cols)} (<{min_n})")
            summary.append({"phase": ph, "n_samples": len(cols), "n_genes": int(gene_mat_z.shape[0]), "written": False})
            continue

        sub = gene_mat_z[cols]  # genes x samples
        out_path = out_dir / f"PHASE__{safe_label(ph)}__n{len(cols)}__zscored.tsv"
        sub.to_csv(out_path, sep="\t")
        print(f"[WRITE] {out_path.name}  shape={sub.shape} (genes x samples)")

        summary.append({"phase": ph, "n_samples": len(cols), "n_genes": int(sub.shape[0]), "written": True})

    summary_df = pd.DataFrame(summary)
    return summary_df

# -----------------------------
# Paths (notebook is in ./preprocessing/burn/)
# -----------------------------
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parents[1]          # BurnInjuries/
BURN_DATA_DIR = PROJECT_ROOT / "preprocessing" / "data" / "GSE182616"

series_path = BURN_DATA_DIR / "Burn_GSE182616_series_matrix.txt"
anno_path   = BURN_DATA_DIR / "BurnAgilentNotation_GPL17077-17467.txt"

out_dir = NOTEBOOK_DIR / "stratified"   # ./preprocessing/burn/stratified

summary_df = run_phase_only_pipeline(
    series_path=series_path,
    anno_path=anno_path,
    out_dir=out_dir,
    min_n=10,     # change if you want
    debug=True,
)

display(summary_df)

[INFO] Loading annotation: /Users/zoeazra/Documents/CLS/Y2/Thesis/Implementation/BurnInjuries/burn_data/BurnAgilentNotation_GPL17077-17467.txt
[INFO] Loading series matrix: /Users/zoeazra/Documents/CLS/Y2/Thesis/Implementation/BurnInjuries/burn_data/Burn_GSE182616_series_matrix.txt

[INFO] phase_bucket counts:
phase_bucket
Acute            513
Proliferation    203
Remodelling       36
None              33
Name: count, dtype: int64
[DEBUG] Final gene matrix: (32080, 785) (genes x samples)
[SAVED] /Users/zoeazra/Documents/CLS/Y2/Thesis/Implementation/BurnInjuries/preprocessing/burn/stratified/burn_sample_metadata_phase.tsv
[WRITE] PHASE__Acute__n513__zscored.tsv  shape=(32080, 513) (genes x samples)
[WRITE] PHASE__Proliferation__n203__zscored.tsv  shape=(32080, 203) (genes x samples)
[WRITE] PHASE__Remodelling__n36__zscored.tsv  shape=(32080, 36) (genes x samples)


,phase,n_samples,n_genes,written
0,Acute,513,32080,True
1,Proliferation,203,32080,True
2,Remodelling,36,32080,True


Now we filter then genes for each group

In [24]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

# =============================
# CONFIG SWITCH
# =============================
GROUPING = "phase"   # "phase" or "tbsa"

# Mean filter
USE_MEAN_FILTER = True
MEAN_KEEP_QUANTILE = 0.90
MIN_MEAN = None

# Mean-variance filter
USE_MV_FILTER = True
MV_QUANTILE_KEEP = 0.90

# =============================
# utilities: parse group names
# =============================
def parse_group_name_tbsa_only(stem: str):
    parts = stem.split("__")
    if len(parts) < 3 or parts[0] != "TBSA":
        return None
    bucket = parts[1]
    m = re.search(r"__n(\d+)", stem)
    if not m:
        return None
    return bucket, int(m.group(1))

def parse_group_name_phase_only(stem: str):
    parts = stem.split("__")
    if len(parts) < 3 or parts[0] != "PHASE":
        return None
    bucket = parts[1]
    m = re.search(r"__n(\d+)", stem)
    if not m:
        return None
    return bucket, int(m.group(1))

def parse_group_name(stem: str, grouping: str):
    if grouping == "tbsa":
        return parse_group_name_tbsa_only(stem)
    if grouping == "phase":
        return parse_group_name_phase_only(stem)
    raise ValueError("grouping must be 'tbsa' or 'phase'")


# =============================
# loaders
# =============================
def load_burn_groups(preprocessed_dir: Path, grouping: str) -> dict[str, pd.DataFrame]:
    groups = {}
    for fp in sorted(preprocessed_dir.glob("*.tsv")):
        if fp.name.startswith("burn_sample_metadata"):
            continue

        parsed = parse_group_name(fp.stem, grouping)
        if parsed is None:
            continue

        X = pd.read_csv(fp, sep="\t", index_col=0)
        X.index = X.index.astype(str)
        X.columns = X.columns.astype(str)
        X = X.apply(pd.to_numeric, errors="coerce").fillna(0.0)

        groups[fp.stem] = X

    if not groups:
        raise RuntimeError(f"No {grouping}-group .tsv files found in {preprocessed_dir}")
    return groups


def gse37069_control_gene_set(control_genes_tsv: Path) -> set[str]:
    """
    Reads the preprocessed GSE37069 controls genes x samples file
    and returns the gene set for intersection with GSE182616.
    """
    if not control_genes_tsv.exists():
        raise FileNotFoundError(f"GSE37069 control genes file not found: {control_genes_tsv}")

    df = pd.read_csv(control_genes_tsv, sep="\t", index_col=0, usecols=[0])
    return set(df.index.astype(str))


def load_burn_metadata(meta_path: Path) -> pd.DataFrame:
    if not meta_path.exists():
        raise FileNotFoundError(f"Metadata file not found: {meta_path}")
    meta = pd.read_csv(meta_path, sep="\t", index_col=0)
    meta.index = meta.index.astype(str)
    return meta


# =============================
# filters
# =============================
def mean_expression_keep_index(X: pd.DataFrame, keep_quantile: float = 0.90, min_mean=None):
    mu = X.mean(axis=1)

    if min_mean is not None:
        return mu.index[mu >= float(min_mean)]

    if not (0.0 < keep_quantile <= 1.0):
        raise ValueError("keep_quantile must be in (0, 1].")
    thr = np.quantile(mu.values, float(keep_quantile))
    return mu.index[mu >= thr]

def sanitize_metadata_for_regression(meta: pd.DataFrame) -> pd.DataFrame:
    """
    Clean metadata once outside the loop.
    """
    meta = meta.copy()

    if "age" not in meta.columns:
        raise KeyError(f"Metadata is missing 'age' column. Columns: {meta.columns.tolist()}")
    if "gender" not in meta.columns:
        raise KeyError(f"Metadata is missing 'gender' column. Columns: {meta.columns.tolist()}")

    meta["age"] = pd.to_numeric(meta["age"], errors="coerce")
    meta["age"] = meta["age"].fillna(meta["age"].median())

    meta["gender"] = (
        meta["gender"].astype(str)
        .replace({"nan": "Unknown", "None": "Unknown"})
        .fillna("Unknown")
    )

    return meta


def regress_out_age_gender(X: pd.DataFrame, meta_clean: pd.DataFrame) -> pd.DataFrame:
    """
    Assumes metadata already sanitized.
    """
    m = meta_clean.reindex(X.columns).copy()

    G = pd.get_dummies(m["gender"], drop_first=True)

    Z = pd.concat(
        [
            pd.Series(1.0, index=X.columns, name="intercept"),
            m["age"].rename("age"),
            G,
        ],
        axis=1,
    ).astype(float)

    Y = X.values.astype(float)   # genes x n
    Zm = Z.values.astype(float)  # n x k

    beta = (Y @ Zm) @ np.linalg.pinv(Zm.T @ Zm)
    Y_hat = beta @ Zm.T
    R = Y - Y_hat

    return pd.DataFrame(R, index=X.index, columns=X.columns)


def mean_variance_adjust_keep(mu: pd.Series, var: pd.Series, q=0.75) -> pd.Index:
    x = np.log10(mu.astype(float).clip(1e-12).values)
    y = np.log10(var.astype(float).clip(1e-12).values)
    slope, intercept = np.polyfit(x, y, 1)
    resid = y - (slope * x + intercept)
    thr = np.quantile(resid, q)
    keep = resid >= thr
    return mu.index[keep]


# =============================
# runner
# =============================
def run_group_filtering(
    project_root: Path,
    grouping: str,
    burn_preprocessed_dir: str,
    gse37069_control_genes_tsv: str,
    burn_metadata_tsv: str,
    out_dir: str,
):
    project_root = Path(project_root)
    burn_dir = (project_root / burn_preprocessed_dir).resolve()
    control_genes_path = (project_root / gse37069_control_genes_tsv).resolve()
    out_dir = (project_root / out_dir).resolve()
    out_dir.mkdir(parents=True, exist_ok=True)

    meta_path = (project_root / burn_metadata_tsv).resolve()

    print("[INFO] grouping          :", grouping)
    print("[INFO] burn_dir          :", burn_dir)
    print("[INFO] control_genes_tsv :", control_genes_path)
    print("[INFO] out_dir           :", out_dir)

    burn_groups = load_burn_groups(burn_dir, grouping=grouping)
    control_genes = gse37069_control_gene_set(control_genes_path)

    meta = None
    if USE_MV_FILTER:
        meta = load_burn_metadata(meta_path)
        meta = sanitize_metadata_for_regression(meta)
        print(f"[INFO] Loaded metadata: {meta_path} (rows={meta.shape[0]}, cols={meta.shape[1]})")

    print(f"[INFO] {grouping}-group burn files loaded:", len(burn_groups))
    print("[INFO] GSE37069 control genes:", len(control_genes))
    print(f"[INFO] Mean filter: {'ON' if USE_MEAN_FILTER else 'OFF'}")
    if USE_MEAN_FILTER:
        if MIN_MEAN is not None:
            print(f"[INFO]   min_mean = {MIN_MEAN}")
        else:
            print(f"[INFO]   keep_quantile = {MEAN_KEEP_QUANTILE}")
    print(f"[INFO] MV filter  : {'ON' if USE_MV_FILTER else 'OFF'}")
    if USE_MV_FILTER:
        print(f"[INFO]   q = {MV_QUANTILE_KEEP}")

    summary_rows = []

    # convert once for faster intersection checks
    control_genes_index = pd.Index(sorted(control_genes))

    for name, X0 in burn_groups.items():
        bucket, _ = parse_group_name(name, grouping)
        genes_before = int(X0.shape[0])
        n_samples = int(X0.shape[1])

        # 1) GSE182616 ∩ GSE37069 controls
        common = X0.index.intersection(control_genes_index)
        X1 = X0.loc[common]
        genes_after_intersection = int(X1.shape[0])

        # 2) Mean-expression filter (optional)
        if USE_MEAN_FILTER:
            keep_mean = mean_expression_keep_index(
                X1,
                keep_quantile=MEAN_KEEP_QUANTILE,
                min_mean=MIN_MEAN,
            )
            X1m = X1.loc[keep_mean]
        else:
            X1m = X1

        genes_after_mean = int(X1m.shape[0])

        # 3) Mean-variance filter (optional)
        if USE_MV_FILTER:
            R = regress_out_age_gender(X1m, meta)
            mu = X1m.mean(axis=1)
            var_resid = R.var(axis=1, ddof=1)
            keep_mv = mean_variance_adjust_keep(mu, var_resid, q=MV_QUANTILE_KEEP)
            X2 = X1m.loc[keep_mv]
        else:
            X2 = X1m

        genes_after_mv = int(X2.shape[0])

        out_fp = out_dir / f"{name}__filtered.tsv"
        X2.to_csv(out_fp, sep="\t")

        print(f"[WRITE] {out_fp.name}: {genes_before} -> {genes_after_intersection} -> {genes_after_mean} -> {genes_after_mv}")

        row = {
            "group": name,
            "n_samples": n_samples,
            "genes_before": genes_before,
            "genes_after_intersection": genes_after_intersection,
            "genes_after_mean_filter": genes_after_mean,
            "did_mv": USE_MV_FILTER,
            "genes_after_mv_adjust": genes_after_mv,
        }

        if grouping == "tbsa":
            row["tbsa_bucket"] = bucket
        else:
            row["phase"] = bucket

        summary_rows.append(row)

    summary_df = pd.DataFrame(summary_rows)

    sort_cols = ["n_samples"]
    if grouping == "tbsa" and "tbsa_bucket" in summary_df.columns:
        sort_cols = ["tbsa_bucket"] + sort_cols
    if grouping == "phase" and "phase" in summary_df.columns:
        sort_cols = ["phase"] + sort_cols

    summary_df = summary_df.sort_values(sort_cols, ascending=[True] + [False] * (len(sort_cols) - 1))

    summary_out = out_dir / "filter_summary.tsv"
    summary_df.to_csv(summary_out, sep="\t", index=False)
    print(f"[SAVED] {summary_out}")

    return summary_df


# =============================
# RUN
# =============================
PROJECT_ROOT = Path("/Users/zoeazra/Documents/CLS/Y2/Thesis/Implementation/BurnInjuries")

summary_df = run_group_filtering(
    project_root=PROJECT_ROOT,
    grouping=GROUPING,
    burn_preprocessed_dir="preprocessing/burn/stratified/GSE182616",   # GSE182616 grouped files
    gse37069_control_genes_tsv="preprocessing/burn_control/preprocessed/GSE37069_controls__genes_x_samples.tsv",
    burn_metadata_tsv="preprocessing/burn/stratified/GSE182616/burn_sample_metadata_phase.tsv",
    out_dir=f"preprocessing/burn/filtered/GSE182616/{GROUPING}",
)

summary_df

[INFO] grouping          : phase
[INFO] burn_dir          : /Users/zoeazra/Documents/CLS/Y2/Thesis/Implementation/BurnInjuries/preprocessing/burn/stratified/GSE182616
[INFO] control_genes_tsv : /Users/zoeazra/Documents/CLS/Y2/Thesis/Implementation/BurnInjuries/preprocessing/burn_control/preprocessed/GSE37069_controls__genes_x_samples.tsv
[INFO] out_dir           : /Users/zoeazra/Documents/CLS/Y2/Thesis/Implementation/BurnInjuries/preprocessing/burn/filtered/GSE182616/phase
[INFO] Loaded metadata: /Users/zoeazra/Documents/CLS/Y2/Thesis/Implementation/BurnInjuries/preprocessing/burn/stratified/GSE182616/burn_sample_metadata_phase.tsv (rows=785, cols=10)
[INFO] phase-group burn files loaded: 3
[INFO] GSE37069 control genes: 20441
[INFO] Mean filter: ON
[INFO]   keep_quantile = 0.9
[INFO] MV filter  : ON
[INFO]   q = 0.9
[WRITE] PHASE__Acute__n513__zscored__filtered.tsv: 32080 -> 16340 -> 1634 -> 164
[WRITE] PHASE__Proliferation__n203__zscored__filtered.tsv: 32080 -> 16340 -> 1634 -> 164
[

,group,n_samples,genes_before,genes_after_intersection,genes_after_mean_filter,did_mv,genes_after_mv_adjust,phase
0,PHASE__Acute__n513__zscored,513,32080,16340,1634,True,164,Acute
1,PHASE__Proliferation__n203__zscored,203,32080,16340,1634,True,164,Proliferation
2,PHASE__Remodelling__n36__zscored,36,32080,16340,1634,True,164,Remodelling


# GSE37069 data (new burn data)

First we have to preprocess the data and annotate the probes with the correct genes

In [28]:
from pathlib import Path
from io import StringIO
import re
import numpy as np
import pandas as pd

# -----------------------------
# Readers
# -----------------------------
def read_affy_gpl570_annotation(path):
    """
    Reads GPL570 annotation TXT with leading '#' comment header.
    Keeps only the columns we need:
      - ID (probe set)
      - Gene Symbol (gene symbol)
      - SPOT_ID (controls indicator)

    IMPORTANT:
      Affymetrix sometimes stores multi-mapped symbols like:
          CA5B /// CA5BP1
      We use Option A: keep only the first symbol:
          CA5B
    """
    path = Path(path)

    df = pd.read_csv(path, sep="\t", comment="#", dtype=str, low_memory=False)
    df.columns = [c.strip() for c in df.columns]

    for c in df.columns:
        df[c] = df[c].astype(str).str.strip().str.strip('"')
        df.loc[df[c].isin(["nan", "NaN", "None"]), c] = ""

    required = ["ID", "Gene Symbol"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise KeyError(f"Annotation missing columns {missing}. Found columns: {df.columns.tolist()[:30]}")

    if "SPOT_ID" not in df.columns:
        df["SPOT_ID"] = ""

    # ---- NORMALIZE GENE SYMBOLS ----
    # Option A: keep first symbol before "///"
    df["Gene Symbol"] = (
        df["Gene Symbol"]
        .fillna("")
        .astype(str)
        .str.split("///").str[0]
        .str.strip()
    )

    return df


def read_geo_series_matrix_full(path):
    """
    Returns:
      - metadata_df with columns: key, values(list[str])
      - expr_df: ID_REF + GSM columns (strings)
    """
    path = Path(path)
    meta_rows = []
    table_lines = []
    in_table = False

    with path.open("r", encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.rstrip("\n")
            if line.startswith("!series_matrix_table_begin"):
                in_table = True
                continue
            if line.startswith("!series_matrix_table_end"):
                in_table = False
                continue

            if in_table:
                table_lines.append(line)
            elif line.startswith("!"):
                meta_rows.append(line)

    meta_parsed = []
    for row in meta_rows:
        parts = row.split("\t")
        key = parts[0].lstrip("!")
        values = parts[1:]
        meta_parsed.append((key, values))

    metadata_df = pd.DataFrame(
        {"key": [k for k, _ in meta_parsed], "values": [v for _, v in meta_parsed]}
    )

    if not table_lines:
        raise ValueError("No series matrix table found between begin/end markers.")

    expr_df = pd.read_csv(StringIO("\n".join(table_lines)), sep="\t", dtype=str, low_memory=False)
    expr_df.columns = [c.strip().strip('"') for c in expr_df.columns]
    for c in expr_df.columns:
        expr_df[c] = expr_df[c].astype(str).str.strip().str.strip('"')
        expr_df.loc[expr_df[c].isin(["nan", "NaN", "None"]), c] = ""

    return metadata_df, expr_df

# -----------------------------
# Metadata parsing helpers
# -----------------------------
def parse_kv_cell(s):
    if s is None:
        return None
    s = str(s).strip().strip('"')
    if ":" not in s:
        return None
    k, v = s.split(":", 1)
    return k.strip().lower(), v.strip()

def to_float(x):
    try:
        if x is None:
            return None
        xs = str(x).strip()
        if xs.lower() in {"na", "nan", ""}:
            return None
        xs = xs.replace("%", "")
        return float(xs)
    except Exception:
        return None

# -----------------------------
# Time + Phase binning
# -----------------------------
def parse_time_to_hours(raw):
    if raw is None:
        return None
    t = str(raw).strip().strip('"').strip().lower()
    t = t.replace("_", "").replace(" ", "")

    m = re.match(r"^hr(\d+)$", t)
    if m:
        return float(m.group(1))

    m = re.match(r"^d(\d+)$", t)
    if m:
        return 24.0 * float(m.group(1))

    return None


def extract_first_float(x):
    if x is None:
        return None
    s = str(x).strip().strip('"')
    if s == "" or s.lower() in {"na", "n/a", "nan", "none"}:
        return None
    m = re.search(r"[-+]?\d*\.?\d+", s)
    if not m:
        return None
    try:
        return float(m.group(0))
    except Exception:
        return None

def assign_time_bin(time_hours):
    if time_hours is None:
        return None
    if time_hours == 0:
        return "T0"
    if 1 <= time_hours < 24:
        return "Early"
    if 24 <= time_hours < 96:
        return "Mid"
    if 96 <= time_hours < 168:
        return "Late"
    if time_hours >= 169:
        return "FollowUp"
    return "Other"

def assign_phase_bucket(time_bin):
    if time_bin is None:
        return None
    if time_bin in {"T0", "Early", "Mid"}:
        return "Acute"
    if time_bin in {"Late"}:
        return "Proliferation"
    if time_bin in {"FollowUp"}:
        return "Remodelling"
    return None

def assign_age_bucket(age_years):
    if age_years is None or (isinstance(age_years, float) and np.isnan(age_years)):
        return None
    try:
        a = float(age_years)
    except Exception:
        return None

    if 0 <= a <= 10:
        return "Child"
    if 11 <= a <= 17:
        return "Adolescent"
    if 18 <= a <= 40:
        return "YngAdult"
    if 41 <= a <= 64:
        return "MidAdult"
    if 65 <= a <= 85:
        return "Elder"
    return "OtherAge"

def extract_sample_metadata(meta_df):
    geo_acc_rows = meta_df[meta_df["key"].str.lower() == "sample_geo_accession"]
    if geo_acc_rows.empty:
        raise ValueError("Could not find '!Sample_geo_accession' in series matrix metadata.")
    geo_samples = [str(x).strip().strip('"') for x in geo_acc_rows.iloc[0]["values"]]
    per_sample = {gsm: {} for gsm in geo_samples}

    char_rows = meta_df[
        meta_df["key"].str.lower().isin(["sample_characteristics_ch1", "sample_characteristics_ch2"])
    ]

    for _, row in char_rows.iterrows():
        values = row["values"]
        n = min(len(values), len(geo_samples))
        for j in range(n):
            gsm = geo_samples[j]
            kv = parse_kv_cell(values[j])
            if kv is None:
                continue
            k, v = kv
            per_sample[gsm][k] = v

    meta = pd.DataFrame.from_dict(per_sample, orient="index")
    meta.index.name = "sample_id"

    if "age" in meta.columns:
        meta["age_years"] = meta["age"].apply(extract_first_float)
    else:
        meta["age_years"] = np.nan

    meta["age_bucket"] = meta["age_years"].apply(assign_age_bucket)

    if "hours_since_injury" in meta.columns:
        meta["time_point_raw"] = meta["hours_since_injury"]
    else:
        meta["time_point_raw"] = meta.get("time point", None)

    meta["time_hours"] = meta["time_point_raw"].apply(extract_first_float)
    meta["time_bin"] = meta["time_hours"].apply(assign_time_bin)
    meta["phase_bucket"] = meta["time_bin"].apply(assign_phase_bucket)

    meta["is_control"] = meta["time_hours"].isna()

    return meta

def safe_label(x):
    x = str(x)
    x = re.sub(r"\s+", "_", x.strip())
    x = re.sub(r"[^A-Za-z0-9_\-]+", "", x)
    return x

# -----------------------------
# Expression preprocessing
# -----------------------------
def preprocess_burn_affy_gpl570(
    expr_df,
    anno_df,
    probe_col_expr="ID_REF",
    probe_col_anno="ID",
    gene_col_anno="Gene Symbol",
    control_col_anno="SPOT_ID",
    collapse="var",
    winsor_q=0.01,
    min_non_na_frac=0.95,
    min_var=1e-8,
    debug=True,
):
    expr = expr_df.copy()
    anno = anno_df.copy()

    if probe_col_expr not in expr.columns:
        raise ValueError(f"Expected '{probe_col_expr}' in expression table; got {expr.columns[:20].tolist()}")
    if probe_col_anno not in anno.columns:
        raise ValueError(f"Expected '{probe_col_anno}' in annotation table; got {anno.columns[:20].tolist()}")
    if gene_col_anno not in anno.columns:
        raise ValueError(f"Expected '{gene_col_anno}' in annotation table; got {anno.columns[:20].tolist()}")

    expr = expr.rename(columns={probe_col_expr: "probe_id"})
    anno = anno.rename(columns={probe_col_anno: "probe_id", gene_col_anno: "gene_symbol"})
    if control_col_anno in anno.columns:
        anno = anno.rename(columns={control_col_anno: "control_type"})
    else:
        anno["control_type"] = ""

    keep_anno = ["probe_id", "gene_symbol", "control_type"]
    anno = anno[keep_anno].copy()

    expr["probe_id"] = expr["probe_id"].astype(str).str.strip().str.strip('"')
    anno["probe_id"] = anno["probe_id"].astype(str).str.strip().str.strip('"')
    anno["gene_symbol"] = anno["gene_symbol"].fillna("").astype(str).str.strip()
    anno["control_type"] = anno["control_type"].fillna("").astype(str).str.strip()

    # optional but helpful normalization
    anno["gene_symbol"] = anno["gene_symbol"].str.upper()

    merged = expr.merge(anno, on="probe_id", how="left")

    meta_cols = {"probe_id", "gene_symbol", "control_type"}
    sample_cols = [c for c in merged.columns if c not in meta_cols]

    for c in sample_cols:
        merged[c] = pd.to_numeric(merged[c], errors="coerce")

    merged["gene_symbol"] = merged["gene_symbol"].fillna("").astype(str).str.strip()
    merged = merged[merged["gene_symbol"].str.len() > 0]

    ct = merged["control_type"].fillna("").astype(str).str.strip()
    is_affx = merged["probe_id"].astype(str).str.upper().str.startswith("AFFX")
    merged = merged[(ct == "") & (~is_affx)]

    non_na_frac = merged[sample_cols].notna().mean(axis=1)
    merged = merged[non_na_frac >= min_non_na_frac]
    if merged.empty:
        raise RuntimeError("All rows were filtered out after missingness/controls filtering.")

    X = merged[["gene_symbol"] + sample_cols].copy()

    if collapse == "mean":
        gene_mat = X.groupby("gene_symbol")[sample_cols].mean()
    elif collapse == "median":
        gene_mat = X.groupby("gene_symbol")[sample_cols].median()
    elif collapse == "var":
        X["_var"] = X[sample_cols].var(axis=1, ddof=1)
        idx = X.groupby("gene_symbol")["_var"].idxmax()
        gene_mat = X.loc[idx].set_index("gene_symbol")[sample_cols]
    else:
        raise ValueError("collapse must be one of: var, mean, median")

    v = gene_mat.var(axis=1, ddof=1).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    gene_mat = gene_mat.loc[v > min_var]

    lo = gene_mat.quantile(winsor_q, axis=1)
    hi = gene_mat.quantile(1 - winsor_q, axis=1)
    gene_mat = gene_mat.apply(lambda row: row.clip(lo[row.name], hi[row.name]), axis=1)

    mu = gene_mat.mean(axis=1)
    sd = gene_mat.std(axis=1, ddof=1).replace(0, np.nan)
    gene_mat_z = gene_mat.sub(mu, axis=0).div(sd, axis=0).dropna(axis=0)

    if debug:
        print("[DEBUG] Final gene matrix:", gene_mat_z.shape, "(genes x samples)")

    return gene_mat_z

def run_phase_only_pipeline(series_path, anno_path, out_dir, min_n=10, debug=True):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    print(f"[INFO] Loading annotation: {anno_path}")
    anno_df = read_affy_gpl570_annotation(anno_path)

    print(f"[INFO] Loading series matrix: {series_path}")
    meta_df, expr_df = read_geo_series_matrix_full(series_path)

    meta = extract_sample_metadata(meta_df)

    gene_mat_z = preprocess_burn_affy_gpl570(
        expr_df=expr_df,
        anno_df=anno_df,
        probe_col_expr="ID_REF",
        probe_col_anno="ID",
        gene_col_anno="Gene Symbol",
        control_col_anno="SPOT_ID",
        min_non_na_frac=0.95,
        debug=debug,
    )

    # normalize gene index consistently
    gene_mat_z.index = gene_mat_z.index.astype(str).str.strip().str.upper()

    sample_cols = list(gene_mat_z.columns.astype(str))
    meta_aligned = meta.loc[meta.index.intersection(sample_cols)].copy()
    meta_aligned = meta_aligned.reindex(sample_cols)

    out_meta = out_dir / "burn_sample_metadata_phase.tsv"
    meta_aligned.to_csv(out_meta, sep="\t")
    print(f"[SAVED] {out_meta}")

    age_min = pd.to_numeric(meta_aligned["age_years"], errors="coerce").min()
    age_max = pd.to_numeric(meta_aligned["age_years"], errors="coerce").max()
    print(f"[INFO] Age range (years): min={age_min}, max={age_max}")

    age_counts = meta_aligned["age_bucket"].value_counts(dropna=False)
    print("[INFO] Age bucket counts:")
    print(age_counts)

    n_controls = int(meta_aligned["is_control"].sum())
    print(f"[INFO] Control samples (hours_since_injury missing): {n_controls}")

    meta_phase = meta_aligned.dropna(subset=["phase_bucket"]).copy()
    meta_phase = meta_phase[meta_phase["phase_bucket"].isin(["Acute", "Proliferation", "Remodelling"])]

    print(f"[INFO] Non-control samples usable for phase binning: {meta_phase.shape[0]}")

    keep_phases = ["Acute", "Proliferation", "Remodelling"]
    summary = []

    for phase in keep_phases:
        cols = meta_phase.index[meta_phase["phase_bucket"] == phase].tolist()

        if len(cols) < min_n:
            print(f"[SKIP] {phase}: n={len(cols)} (<{min_n})")
            summary.append({"phase": phase, "n_samples": len(cols), "n_genes": int(gene_mat_z.shape[0]), "written": False})
            continue

        sub = gene_mat_z[cols]
        out_path = out_dir / f"PHASE__{safe_label(phase)}__n{len(cols)}__zscored.tsv"
        sub.to_csv(out_path, sep="\t")
        print(f"[WRITE] {out_path.name}  shape={sub.shape} (genes x samples)")

        summary.append({"phase": phase, "n_samples": len(cols), "n_genes": int(sub.shape[0]), "written": True})

    summary_df = pd.DataFrame(summary)
    summary_df["age_min"] = age_min
    summary_df["age_max"] = age_max
    summary_df["n_controls"] = n_controls

    return summary_df

# -----------------------------
# Paths
# -----------------------------
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parents[1]
BURN_DATA_DIR = PROJECT_ROOT / "preprocessing" / "data" / "GSE37069"

series_path = BURN_DATA_DIR / "Burn_GSE37069_series_matrix.txt"
anno_path   = BURN_DATA_DIR / "Annotation_GPL570-55999.txt"

out_dir = NOTEBOOK_DIR / "stratified" / "GSE37069"

summary_df = run_phase_only_pipeline(
    series_path=series_path,
    anno_path=anno_path,
    out_dir=out_dir,
    min_n=1,
    debug=True,
)

display(summary_df)

[INFO] Loading annotation: /Users/zoeazra/Documents/CLS/Y2/Thesis/Implementation/BurnInjuries/preprocessing/data/GSE37069/Annotation_GPL570-55999.txt
[INFO] Loading series matrix: /Users/zoeazra/Documents/CLS/Y2/Thesis/Implementation/BurnInjuries/preprocessing/data/GSE37069/Burn_GSE37069_series_matrix.txt
[DEBUG] Final gene matrix: (19923, 590) (genes x samples)
[SAVED] /Users/zoeazra/Documents/CLS/Y2/Thesis/Implementation/BurnInjuries/preprocessing/burn/stratified/GSE37069/burn_sample_metadata_phase.tsv
[INFO] Age range (years): min=0.0, max=86.0
[INFO] Age bucket counts:
age_bucket
YngAdult      194
Child         167
MidAdult      141
Adolescent     57
Elder          28
OtherAge        3
Name: count, dtype: int64
[INFO] Control samples (hours_since_injury missing): 37
[INFO] Non-control samples usable for phase binning: 553
[WRITE] PHASE__Acute__n239__zscored.tsv  shape=(19923, 239) (genes x samples)
[WRITE] PHASE__Proliferation__n32__zscored.tsv  shape=(19923, 32) (genes x samples)


,phase,n_samples,n_genes,written,age_min,age_max,n_controls
0,Acute,239,19923,True,0.0,86.0,37
1,Proliferation,32,19923,True,0.0,86.0,37
2,Remodelling,282,19923,True,0.0,86.0,37


Since there are also children in this dataset, we can filter them out first and then start filtering the genes.

In [29]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

# =============================
# CONFIG SWITCHES
# =============================
GROUPING = "phase"          
ADULT_MIN_AGE = 18

USE_MEAN_FILTER = True      # <-- toggle mean filter on/off
MEAN_KEEP_QUANTILE = 0.90   # keep top X% by mean within group
MIN_MEAN = None             # optional absolute mean cutoff (overrides quantile)

USE_MV_FILTER = True        # <-- toggle MV filter on/off
MV_QUANTILE_KEEP = 0.98     # keep top X% by MV residual

# paths
PROJECT_ROOT = Path("/Users/zoeazra/Documents/CLS/Y2/Thesis/Implementation/BurnInjuries")
BURN_DIR = PROJECT_ROOT / "preprocessing/burn/stratified/GSE37069"
META_TSV = BURN_DIR / "burn_sample_metadata_phase.tsv"   # adjust if your metadata file is named differently
OUT_DIR = PROJECT_ROOT / f"preprocessing/burn/filtered/GSE37069/{GROUPING}"
OUT_DIR.mkdir(parents=True, exist_ok=True)


# -----------------------------
# utilities: parse group names
# -----------------------------
def parse_group_name_tbsa_only(stem: str):
    parts = stem.split("__")
    if len(parts) < 3 or parts[0] != "TBSA":
        return None
    bucket = parts[1]
    m = re.search(r"__n(\d+)", stem)
    if not m:
        return None
    return bucket, int(m.group(1))

def parse_group_name_phase_only(stem: str):
    parts = stem.split("__")
    if len(parts) < 3 or parts[0] != "PHASE":
        return None
    bucket = parts[1]
    m = re.search(r"__n(\d+)", stem)
    if not m:
        return None
    return bucket, int(m.group(1))

def parse_group_name(stem: str, grouping: str):
    if grouping == "tbsa":
        return parse_group_name_tbsa_only(stem)
    if grouping == "phase":
        return parse_group_name_phase_only(stem)
    raise ValueError("grouping must be 'tbsa' or 'phase'")


# -----------------------------
# loaders
# -----------------------------
def load_burn_groups(preprocessed_dir: Path, grouping: str) -> dict[str, pd.DataFrame]:
    groups = {}
    for fp in sorted(preprocessed_dir.glob("*.tsv")):
        if fp.name.startswith("burn_sample_metadata"):
            continue
        parsed = parse_group_name(fp.stem, grouping)
        if parsed is None:
            continue

        X = pd.read_csv(fp, sep="\t", index_col=0)
        X.index = X.index.astype(str)
        X.columns = X.columns.astype(str)
        X = X.apply(pd.to_numeric, errors="coerce")  # keep NaN (don’t fill with 0 here)
        groups[fp.stem] = X

    if not groups:
        raise RuntimeError(f"No {grouping}-group .tsv files found in {preprocessed_dir}")
    return groups

def load_metadata(meta_path: Path) -> pd.DataFrame:
    if not meta_path.exists():
        raise FileNotFoundError(f"Metadata file not found: {meta_path}")
    meta = pd.read_csv(meta_path, sep="\t", index_col=0)
    meta.index = meta.index.astype(str)
    return meta


# -----------------------------
# filters
# -----------------------------
def mean_expression_keep_index(X: pd.DataFrame, keep_quantile: float = 0.90, min_mean=None):
    mu = X.mean(axis=1, skipna=True)

    if min_mean is not None:
        return mu.index[mu >= float(min_mean)]

    if not (0.0 < keep_quantile <= 1.0):
        raise ValueError("keep_quantile must be in (0, 1].")
    thr = np.quantile(mu.values, float(keep_quantile))
    return mu.index[mu >= thr]

def regress_out_age_gender(X: pd.DataFrame, meta: pd.DataFrame) -> pd.DataFrame:
    """
    Expects metadata columns: 'age' and 'sex' OR 'gender'
    """
    m = meta.reindex(X.columns).copy()

    if "age" not in m.columns:
        raise KeyError(f"Metadata missing 'age'. Columns: {m.columns.tolist()}")

    sex_col = "gender" if "gender" in m.columns else ("sex" if "sex" in m.columns else None)
    if sex_col is None:
        raise KeyError(f"Metadata missing 'gender' or 'sex'. Columns: {m.columns.tolist()}")

    m["age"] = pd.to_numeric(m["age"], errors="coerce")
    m["age"] = m["age"].fillna(m["age"].median())

    m[sex_col] = (
        m[sex_col].astype(str)
        .replace({"nan": "Unknown", "None": "Unknown"})
        .fillna("Unknown")
    )
    G = pd.get_dummies(m[sex_col], drop_first=True)

    Z = pd.concat(
        [
            pd.Series(1.0, index=X.columns, name="intercept"),
            m["age"].rename("age"),
            G,
        ],
        axis=1,
    ).astype(float)

    Y = X.values.astype(float)   # genes x n
    Zm = Z.values.astype(float)  # n x k
    beta = (Y @ Zm) @ np.linalg.pinv(Zm.T @ Zm)  # genes x k
    Y_hat = beta @ Zm.T                          # genes x n
    R = Y - Y_hat
    return pd.DataFrame(R, index=X.index, columns=X.columns)

def mean_variance_adjust_keep(mu: pd.Series, var: pd.Series, q=0.75) -> pd.Index:
    x = np.log10(mu.astype(float).clip(1e-12).values)
    y = np.log10(var.astype(float).clip(1e-12).values)
    slope, intercept = np.polyfit(x, y, 1)
    resid = y - (slope * x + intercept)
    thr = np.quantile(resid, q)
    keep = resid >= thr
    return mu.index[keep]


# -----------------------------
# runner
# -----------------------------
def run_group_filtering_adults_only():
    print("[INFO] burn_dir:", BURN_DIR)
    print("[INFO] grouping:", GROUPING)
    print("[INFO] out_dir :", OUT_DIR)

    burn_groups = load_burn_groups(BURN_DIR, grouping=GROUPING)
    meta = load_metadata(META_TSV)

    # adult filter mask (global)
    meta["age_num"] = pd.to_numeric(meta["age"], errors="coerce")
    adult_samples = meta.index[meta["age_num"] >= ADULT_MIN_AGE].astype(str).tolist()

    print(f"[INFO] Total metadata samples: {meta.shape[0]}")
    print(f"[INFO] Adult samples (age >= {ADULT_MIN_AGE}): {len(adult_samples)}")
    print(f"[INFO] USE_MEAN_FILTER={USE_MEAN_FILTER} (q={MEAN_KEEP_QUANTILE}, min_mean={MIN_MEAN})")
    print(f"[INFO] USE_MV_FILTER={USE_MV_FILTER} (q={MV_QUANTILE_KEEP})")

    summary_rows = []

    for name, X0 in burn_groups.items():
        bucket, _ = parse_group_name(name, GROUPING)

        genes_before = int(X0.shape[0])
        n_samples_before = int(X0.shape[1])

        # 0) Adult-only samples (and drop columns not in metadata/adult set)
        cols = [c for c in X0.columns if (c in adult_samples)]
        X = X0[cols]
        n_samples_adult = int(X.shape[1])

        # if a group becomes empty after adult filtering, skip writing
        if n_samples_adult == 0:
            print(f"[SKIP] {name}: 0 adult samples after filtering")
            row = dict(
                group=name,
                n_samples=n_samples_before,
                n_samples_adult=n_samples_adult,
                genes_before=genes_before,
                genes_after_mean_filter=np.nan,
                did_mv=USE_MV_FILTER,
                genes_after_mv_adjust=np.nan,
            )
            if GROUPING == "tbsa":
                row["tbsa_bucket"] = bucket
            else:
                row["phase"] = bucket
            summary_rows.append(row)
            continue

        # 1) Mean filter (optional)
        if USE_MEAN_FILTER:
            keep_mean = mean_expression_keep_index(X, keep_quantile=MEAN_KEEP_QUANTILE, min_mean=MIN_MEAN)
            Xm = X.loc[keep_mean]
        else:
            Xm = X

        genes_after_mean = int(Xm.shape[0])

        # 2) MV filter (optional)
        if USE_MV_FILTER:
            R = regress_out_age_gender(Xm, meta)
            mu = Xm.mean(axis=1, skipna=True)
            var_resid = R.var(axis=1, ddof=1)
            keep_mv = mean_variance_adjust_keep(mu, var_resid, q=MV_QUANTILE_KEEP)
            X2 = Xm.loc[keep_mv]
        else:
            X2 = Xm

        genes_after_mv = int(X2.shape[0])

        out_fp = OUT_DIR / f"{name}__adult__filtered.tsv"
        X2.to_csv(out_fp, sep="\t")
        print(f"[WRITE] {out_fp.name}: samples {n_samples_before}->{n_samples_adult}, genes {genes_before}->{genes_after_mean}->{genes_after_mv}")

        row = dict(
            group=name,
            n_samples=n_samples_before,
            n_samples_adult=n_samples_adult,
            genes_before=genes_before,
            genes_after_mean_filter=genes_after_mean,
            did_mv=USE_MV_FILTER,
            genes_after_mv_adjust=genes_after_mv,
        )
        if GROUPING == "tbsa":
            row["tbsa_bucket"] = bucket
        else:
            row["phase"] = bucket
        summary_rows.append(row)

    summary_df = pd.DataFrame(summary_rows)

    sort_cols = ["n_samples_adult"]
    if GROUPING == "tbsa" and "tbsa_bucket" in summary_df.columns:
        sort_cols = ["tbsa_bucket"] + sort_cols
    if GROUPING == "phase" and "phase" in summary_df.columns:
        sort_cols = ["phase"] + sort_cols

    summary_df = summary_df.sort_values(sort_cols, ascending=[True] + [False]*(len(sort_cols)-1))
    summary_out = OUT_DIR / "filter_summary.tsv"
    summary_df.to_csv(summary_out, sep="\t", index=False)
    print(f"[SAVED] {summary_out}")

    return summary_df


summary_df = run_group_filtering_adults_only()
summary_df

[INFO] burn_dir: /Users/zoeazra/Documents/CLS/Y2/Thesis/Implementation/BurnInjuries/preprocessing/burn/stratified/GSE37069
[INFO] grouping: phase
[INFO] out_dir : /Users/zoeazra/Documents/CLS/Y2/Thesis/Implementation/BurnInjuries/preprocessing/burn/filtered/GSE37069/phase
[INFO] Total metadata samples: 590
[INFO] Adult samples (age >= 18): 366
[INFO] USE_MEAN_FILTER=True (q=0.9, min_mean=None)
[INFO] USE_MV_FILTER=True (q=0.98)
[WRITE] PHASE__Acute__n239__zscored__adult__filtered.tsv: samples 239->130, genes 19923->1993->40
[WRITE] PHASE__Proliferation__n32__zscored__adult__filtered.tsv: samples 32->23, genes 19923->1993->40
[WRITE] PHASE__Remodelling__n282__zscored__adult__filtered.tsv: samples 282->176, genes 19923->1993->40
[SAVED] /Users/zoeazra/Documents/CLS/Y2/Thesis/Implementation/BurnInjuries/preprocessing/burn/filtered/GSE37069/phase/filter_summary.tsv


,group,n_samples,n_samples_adult,genes_before,genes_after_mean_filter,did_mv,genes_after_mv_adjust,phase
0,PHASE__Acute__n239__zscored,239,130,19923,1993,True,40,Acute
1,PHASE__Proliferation__n32__zscored,32,23,19923,1993,True,40,Proliferation
2,PHASE__Remodelling__n282__zscored,282,176,19923,1993,True,40,Remodelling


# GSE182616 with control from GSE37069